In [3]:
import os
os.environ.pop("PYTHONPATH", None)

import sys
sys.path = [
    p for p in sys.path
    if "/opt/software-current/" not in p
]

import regex
print("regex:", regex.__file__)

regex: /mnt/home/linpaul1/Documents/SPROUT/.venv/lib/python3.13/site-packages/regex/__init__.py


In [4]:
import torch
from transformers import AutoModel, AutoProcessor
from PIL import Image
import torch.nn.functional as F
import pandas as pd

MODEL_NAME = "google/siglip2-base-patch16-224"

device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModel.from_pretrained(MODEL_NAME).to(device).eval()
processor = AutoProcessor.from_pretrained(MODEL_NAME)

/mnt/home/linpaul1/Documents/SPROUT/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.
Loading weights: 100%|██████████| 408/408 [00:00<00:00, 16462.81it/s]


In [8]:
def embed_images_batch(image_paths, batch_size: int = 128):
    embeddings = []


    for start in range(0, len(image_paths), batch_size):
        batch_paths = image_paths[start:start + batch_size]

        images = []
        for path in batch_paths:
            with Image.open(path) as img:
                images.append(img.convert("RGB"))

        inputs = processor(images=images, return_tensors="pt").to(device)

        with torch.inference_mode():
            output = model.get_image_features(**inputs)
            embedding = output.pooler_output
            embedding = F.normalize(embedding, p=2, dim=-1)

        embeddings.append(embedding)

    # return embedding.squeeze(0)
    return torch.cat(embeddings, dim=0)

def embed_text_batch(texts, batch_size: int = 128):
    embeddings = []

    for start in range(0, len(texts), batch_size):

        inputs = processor(text=texts[start:start + batch_size],
            padding="max_length", return_tensors="pt").to(device)

        with torch.inference_mode():
            output = model.get_text_features(**inputs)
            embedding = output.pooler_output
            embedding = F.normalize(embedding, p=2, dim=-1)

        embeddings.append(embedding)

    return torch.cat(embeddings, dim=0)

def embed_post_batch(df, batch_size=256, image_weight=0.6):
    image_paths = df["image_path"].astype(str).tolist()
    image_embeddings = embed_images_batch(image_paths, batch_size)

    caption_paths = df["caption"].astype(str).tolist()
    caption_embeddings = embed_text_batch(caption_paths, batch_size)

    embeddings = (image_embeddings * image_weight) + (caption_embeddings * (1 * image_weight))

    return embeddings

def embed_collection_batch(df, batch_size=128):
    titles = (
        df["collection_title"]
        .fillna("")
        .astype(str)
        .tolist()
    )

    return embed_text_batch(titles, batch_size)

In [9]:
df = pd.read_parquet("../clip_train.parquet")
collections_df = pd.read_csv("hi/collections.tsv000", sep="\t")

In [10]:
unique_collections = (
    collections_df[
        ["collection_id", "collection_title"]
    ]
    .drop_duplicates("collection_id")
    .reset_index(drop=True)
)

In [11]:
post_embeddings = embed_post_batch(df)

collection_embeddings = embed_collection_batch(
    unique_collections
)

Post embeddings: torch.Size([24997, 768])
Collection embeddings: torch.Size([558146, 768])


In [12]:
photo_ids = df["photo_id"].tolist()

photo_to_idx = {
    photo_id: idx
    for idx, photo_id in enumerate(photo_ids)
}

collection_ids = unique_collections["collection_id"].tolist()

collection_to_idx = {
    collection_id: idx
    for idx, collection_id in enumerate(collection_ids)
}

In [13]:
import numpy as np

edges_df = (
    collections_df[
        ["photo_id", "collection_id"]
    ]
    .drop_duplicates()
)

edges_df = edges_df[
    edges_df["photo_id"].isin(photo_to_idx)
    & edges_df["collection_id"].isin(collection_to_idx)
]

source = edges_df["photo_id"].map(photo_to_idx).to_numpy()
target = edges_df["collection_id"].map(collection_to_idx).to_numpy()

edge_index = torch.tensor(
    np.stack([source, target]),
    dtype=torch.long
)

print("Edge index:", edge_index.shape)

Edge index: torch.Size([2, 2204768])


In [14]:
from torch_geometric.data import HeteroData

data = HeteroData()

# Node features
data["photo"].x = post_embeddings
data["collection"].x = collection_embeddings

# Photo -> Collection
data["photo", "belongs_to", "collection"].edge_index = edge_index

# Collection -> Photo
data["collection", "contains", "photo"].edge_index = edge_index.flip(0)

print(data)

/mnt/home/linpaul1/Documents/SPROUT/.venv/lib/python3.13/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


HeteroData(
  photo={ x=[24997, 768] },
  collection={ x=[558146, 768] },
  (photo, belongs_to, collection)={ edge_index=[2, 2204768] },
  (collection, contains, photo)={ edge_index=[2, 2204768] }
)


In [24]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv

/mnt/home/linpaul1/Documents/SPROUT/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/mnt/home/linpaul1/Documents/SPROUT/.venv/lib/python3.13/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [ ]:
class GraphSAGE(nn.Module):
    def __init__(self, input_dim=512, hidden_dim=256, output_dim=128):
        super().__init__()

        self.layer1 = SAGEConv(512, 256)
        self.layer2 = SAGEConv(256, 128)

    def forward(self, x, edge_index):
        x = F.relu(self.layer1(x, edge_index))
        x = self.layer2(x, edge_index)

        return x

model = GraphSAGE().cuda()

In [ ]:
out = model(features, edge_index)

: 